# Multi-Horizon LOB Training Pipeline

**Full pipeline with checkpoint/recovery support.**

Run cells top-to-bottom. If the session is interrupted, simply re-run from Cell 1 — the checkpoint system will automatically skip already-completed architectures and resume in-progress ones from the last saved epoch.

### Recovery guarantee
- After every epoch, model weights + training state are saved to Drive/disk.
- On restart, completed architectures are detected and skipped.
- In-progress architectures resume from the last completed epoch.
- All paths live in `WEIGHTS_DIR` and `RESULTS_DIR` (on Drive in Colab).
### Accuracy-improvement features (Cells 4b–4c)
- **7 normalization methods** — ZScore, MinMax, Robust, MaxAbs, Quantile, Power, Decimal (set `normalization_method` in Cell 4)
- **SMOTE oversampling** — handles class imbalance (DOWN/UP ≪ STATIONARY); set `enable_smote=True` + `pip install imbalanced-learn`
- **Stationarity transform** — ADF test + fractional differencing for price features; set `enable_stationarity=True`
- **R² metrics** — post-training evaluation of model calibration (Cell 13b)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 1 — Environment Setup (Colab / local)
# Run once per session. Safe to re-run.
# ═══════════════════════════════════════════════════════════════════════════
import os
from pathlib import Path


def is_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


IN_COLAB = is_colab()

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/multi-horizon-ofi')
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR    = str(PROJECT_ROOT / 'data' / 'processed')
WEIGHTS_DIR = str(PROJECT_ROOT / 'model_weights')
RESULTS_DIR = str(PROJECT_ROOT / 'results')

Path(WEIGHTS_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"DATA_DIR not found: {DATA_DIR}. "
        "In Colab, ensure Drive is mounted and the repo is at /content/drive/MyDrive/multi-horizon-ofi"
    )

tickers = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
    and any(f.endswith('.parquet') for f in os.listdir(os.path.join(DATA_DIR, d)))
])

print(f"IN_COLAB    : {IN_COLAB}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR    : {DATA_DIR}")
print(f"WEIGHTS_DIR : {WEIGHTS_DIR}")
print(f"RESULTS_DIR : {RESULTS_DIR}")
print(f"Tickers     : {tickers}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 2 — Imports
# ═══════════════════════════════════════════════════════════════════════════
import gc
import glob
import hashlib
import json
import math
import os
import random
import time
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

print(f"torch={torch.__version__}  cuda={torch.cuda.is_available()}")
print("Imports OK")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 3 — Label Generation
# ═══════════════════════════════════════════════════════════════════════════
"""
Multi-horizon fixed-threshold classification labels.
Labels: 0=DOWN, 1=STATIONARY, 2=UP
"""

DEFAULT_HORIZONS: List[int] = [10, 20, 50, 100]


def _mid_price_array(df: pd.DataFrame, dtype=np.float64) -> np.ndarray:
    bp = df["bid_price_1"].to_numpy(dtype=dtype, copy=False)
    ap = df["ask_price_1"].to_numpy(dtype=dtype, copy=False)
    return (ap + bp) * 0.5


def _smoothed_mid(m: np.ndarray, k: int) -> np.ndarray:
    n = len(m)
    out = np.full(n, np.nan, dtype=m.dtype)
    if k >= n:
        return out
    cumsum = np.cumsum(m, dtype=m.dtype)
    out[: n - k] = (cumsum[k:] - cumsum[:-k]) / k
    return out


def make_fixed_threshold_classification_labels(
    df: pd.DataFrame,
    horizons: List[int] | None = None,
    alpha: float = 0.002,
    use_smoothing: bool = True,
    dtype=np.float64,
) -> pd.DataFrame:
    if horizons is None:
        horizons = DEFAULT_HORIZONS
    m = _mid_price_array(df, dtype)
    n = len(m)
    out = np.empty((n, len(horizons)), dtype=dtype)
    for i, k in enumerate(horizons):
        future_m = _smoothed_mid(m, k) if use_smoothing else np.full(n, np.nan, dtype=dtype)
        if not use_smoothing and k < n:
            future_m[: n - k] = m[k:]
        pct_change = (future_m - m) / m
        labels = np.full(n, np.nan, dtype=dtype)
        labels[pct_change > alpha] = 2
        labels[pct_change < -alpha] = 0
        mid_mask = (~np.isnan(pct_change)) & (np.isnan(labels))
        labels[mid_mask] = 1
        out[:, i] = labels
    cols = [f"label_{k}" for k in horizons]
    return pd.DataFrame(out, index=df.index, columns=cols)


print("Label functions ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 4 — Global Config
# Tune these before training.
# ═══════════════════════════════════════════════════════════════════════════

DEEP_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Raw 10-level LOB columns (40 features per event)
DEEP_RAW_LOB_10_COLS = [
    f"{side}_{field}_{lvl}"
    for lvl in range(1, 11)
    for side, field in (("ask", "price"), ("ask", "size"), ("bid", "price"), ("bid", "size"))
]

DEEP_CONFIG: Dict = {
    # Paths (set by Cell 1)
    "data_dir":    DATA_DIR,
    "weights_dir": WEIGHTS_DIR,
    "results_dir": RESULTS_DIR,

    # Tickers (None → auto-discover)
    "tickers": None,

    # Horizons
    "horizons": [10, 20, 50, 100],

    # Label threshold
    "alpha": 0.00005,

    # Sequence length for windowed input
    "seq_len": 100,

    # Train/eval file split
    "train_file_fraction": 0.8,

    # Subsampling (reduce to fit in Colab RAM)
    "base_subsample":             4,
    "high_pressure_subsample":    8,
    "critical_pressure_subsample": 16,
    "max_rows_per_day_train":  8000,
    "max_rows_per_day_eval":  10000,
    "max_files_per_ticker":       0,   # 0 = all files

    # DataLoader
    "batch_size":   256,
    "num_workers":  0,

    # Optimiser
    "lr":            3e-4,
    "weight_decay":  1e-4,
    "grad_clip":     1.0,
    "amp":           True,    # use AMP if CUDA available

    # Loss
    "loss_mode":       "cb_focal",  # "ce" or "cb_focal"
    "cb_beta":          0.999,
    "cb_min_w":         0.5,
    "cb_max_w":         3.0,
    "cb_eps":           1.0,
    "focal_gamma":      2.0,
    "label_smoothing":  0.02,

    # Training loop
    "seed": 42,

    # Architectures to train (order matters for sequential execution)
    "run_architectures": ["dilated_transformer", "cnn_inception_lstm", "seq2seq_attn"],

    # Result file suffix
    "result_suffix": "_stable_es",

    # ── Normalization ─────────────────────────────────────────────────────
    # Method used to normalise LOB features before model input.
    # Options: "zscore" | "minmax" | "robust" | "maxabs" | "quantile" | "power" | "decimal"
    # "robust"  — best for heavy-tailed LOB data (uses median/IQR)
    # "quantile" — maps to uniform/normal distribution, great for skewed features
    # "maxabs"  — preserves zero entries (good for sparse OFI features)
    "normalization_method": "robust",

    # ── Class Imbalance / SMOTE ───────────────────────────────────────────
    # STATIONARY class typically dominates (60-70%), causing model collapse.
    # enable_smote requires:  pip install imbalanced-learn
    "enable_smote":   False,   # True to apply SMOTE oversampling per day
    "smote_method":   "smote", # "smote" | "borderline_smote" | "adasyn" | "smote_tomek"
    "smote_k":        5,       # k_neighbors for SMOTE
    "smote_min_per_class": 10, # skip days with fewer than this per class

    # ── Stationarity ──────────────────────────────────────────────────────
    # LOB price features are near-random-walks (non-stationary).
    # Enabling transforms them before model input (adds memory but takes time).
    "enable_stationarity": False,  # True to run ADF + fractional differencing
    "stationarity_method": "auto", # "auto" | "diff" | "frac_diff" | "frac_min_d"
    "stationarity_d_fixed": 0.4,   # d used when method="frac_diff"
}

# ─── Training schedule ────────────────────────────────────────────────────
DEEP_MAX_EPOCHS          = 10   # max epochs per architecture
DEEP_EARLY_STOP_PATIENCE = 3    # stop if no improvement for N epochs
DEEP_EARLY_STOP_MIN_DELTA = 1e-4

print(f"Device      : {DEEP_DEVICE}")
print(f"Architectures: {DEEP_CONFIG['run_architectures']}")
print(f"Max epochs  : {DEEP_MAX_EPOCHS}  patience={DEEP_EARLY_STOP_PATIENCE}")
print(f"Config OK")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 4b — Inline Accuracy-Improvement Modules
#
# Provides (inline, no dependency on src/):
#   • Normalization scalers  — ZScore, MinMax, Robust, MaxAbs, Quantile,
#                              PowerTransformer, DecimalScaler
#   • R² metrics             — standard, adjusted, out-of-sample
#   • Stationarity           — ADF test, fractional differencing (FFD)
#   • SMOTE diagnostics      — class distribution, imbalance ratio
#
# Import preference: tries src.* first; falls back to inline definitions.
# ═══════════════════════════════════════════════════════════════════════════
import sys, warnings
from pathlib import Path as _Path

# Ensure project root is on path (works in Colab when repo is mounted)
_project_root = str(PROJECT_ROOT)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

# ─────────────────────────────────────────────────────────────────────────
# A.  Normalization scalers
# ─────────────────────────────────────────────────────────────────────────
try:
    from src.data.preprocessor import (
        ZScoreScaler, MinMaxScaler, RobustScaler,
        MaxAbsScaler, QuantileScaler, PowerTransformer,
        DecimalScaler, normalize_splits,
    )
    print("  [norm] loaded from src.data.preprocessor")
except ImportError:
    # ── Inline fallback ───────────────────────────────────────────────────
    class ZScoreScaler:
        def fit(self, X):
            self.mean_ = X.mean(0); self.std_ = np.where(X.std(0)<1e-8,1.0,X.std(0)); return self
        def transform(self, X): return (X - self.mean_) / self.std_
        def fit_transform(self, X): return self.fit(X).transform(X)

    class MinMaxScaler:
        def fit(self, X):
            self.min_ = X.min(0); self.max_ = X.max(0)
            self.range_ = np.where(self.max_-self.min_<1e-8,1.0,self.max_-self.min_); return self
        def transform(self, X): return (X - self.min_) / self.range_
        def fit_transform(self, X): return self.fit(X).transform(X)

    class RobustScaler:
        def fit(self, X):
            self.median_ = np.median(X,0)
            q75,q25 = np.percentile(X,[75,25],0)
            self.iqr_ = np.where(q75-q25<1e-8,1.0,q75-q25); return self
        def transform(self, X): return (X - self.median_) / self.iqr_
        def fit_transform(self, X): return self.fit(X).transform(X)

    class MaxAbsScaler:
        def fit(self, X):
            self.max_abs_ = np.where(np.abs(X).max(0)<1e-8,1.0,np.abs(X).max(0)); return self
        def transform(self, X): return X / self.max_abs_
        def fit_transform(self, X): return self.fit(X).transform(X)

    class DecimalScaler:
        def fit(self, X):
            max_abs = np.abs(X).max(0)
            self.scale_ = np.power(10, np.ceil(np.log10(np.where(max_abs<1e-8,1.0,max_abs)))); return self
        def transform(self, X): return X / self.scale_
        def fit_transform(self, X): return self.fit(X).transform(X)

    class QuantileScaler:
        def __init__(self, output_distribution='uniform', n_quantiles=1000):
            self.output_distribution = output_distribution
            self.n_quantiles = n_quantiles
        def fit(self, X):
            try:
                from sklearn.preprocessing import QuantileTransformer
                self._sk = QuantileTransformer(output_distribution=self.output_distribution,
                                               n_quantiles=min(self.n_quantiles,len(X)),
                                               random_state=42)
                self._sk.fit(X)
            except ImportError:
                self._sk = None
            return self
        def transform(self, X):
            if self._sk is None: return X
            return self._sk.transform(X)
        def fit_transform(self, X): return self.fit(X).transform(X)

    class PowerTransformer:
        def fit(self, X):
            try:
                from sklearn.preprocessing import PowerTransformer as _PT
                self._sk = _PT(method='yeo-johnson', standardize=True); self._sk.fit(X)
            except ImportError: self._sk = None
            return self
        def transform(self, X):
            if self._sk is None: return X
            return self._sk.transform(X)
        def fit_transform(self, X): return self.fit(X).transform(X)

    _SCALER_MAP = {
        "zscore": ZScoreScaler, "minmax": MinMaxScaler, "robust": RobustScaler,
        "maxabs": MaxAbsScaler, "decimal": DecimalScaler,
        "quantile": lambda: QuantileScaler(output_distribution='uniform'),
        "quantile_normal": lambda: QuantileScaler(output_distribution='normal'),
        "power": PowerTransformer,
    }

    def normalize_splits(X_train, X_val, X_test, method="zscore"):
        factory = _SCALER_MAP.get(method, ZScoreScaler)
        scaler  = factory() if callable(factory) else factory
        X_tr = scaler.fit_transform(X_train.astype(np.float64)).astype(np.float32)
        X_va = scaler.transform(X_val.astype(np.float64)).astype(np.float32)
        X_te = scaler.transform(X_test.astype(np.float64)).astype(np.float32)
        return X_tr, X_va, X_te, scaler

    print("  [norm] using inline scaler definitions")


# ─────────────────────────────────────────────────────────────────────────
# B.  R² metrics
# ─────────────────────────────────────────────────────────────────────────
try:
    from src.metrics.r2 import (
        compute_r2, compute_adjusted_r2, compute_oos_r2,
        compute_all_horizon_r2, print_r2_table,
    )
    print("  [r2]   loaded from src.metrics.r2")
except ImportError:
    def compute_r2(y_true, y_pred):
        ss_res = float(np.sum((y_true - y_pred) ** 2))
        ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
        return 1.0 - ss_res / max(ss_tot, 1e-12)

    def compute_adjusted_r2(y_true, y_pred, n_features: int):
        r2 = compute_r2(y_true, y_pred)
        n  = len(y_true)
        if n <= n_features + 1: return float("nan")
        return 1.0 - (1.0 - r2) * (n - 1) / (n - n_features - 1)

    def compute_oos_r2(y_true, y_pred):
        """Out-of-sample R² vs. historical mean benchmark."""
        ss_res = float(np.sum((y_true - y_pred) ** 2))
        ss_bm  = float(np.sum((y_true - y_true.mean()) ** 2))
        return 1.0 - ss_res / max(ss_bm, 1e-12)

    def compute_all_horizon_r2(Y_true, Y_pred, horizons, n_features=40):
        results = {}
        for i, h in enumerate(horizons):
            yt = Y_true[:, i].astype(np.float64)
            yp = Y_pred[:, i].astype(np.float64)
            results[f"h{h}"] = {
                "r2":      compute_r2(yt, yp),
                "r2_adj":  compute_adjusted_r2(yt, yp, n_features),
                "r2_oos":  compute_oos_r2(yt, yp),
            }
        return results

    def print_r2_table(results, horizons):
        print(f"\n{'='*55}")
        print(f"   {'Horizon':>8}   {'R²':>8}   {'R²_adj':>8}   {'R²_oos':>8}")
        print(f"{'='*55}")
        for h in horizons:
            d = results.get(f"h{h}", {})
            print(f"   {'k='+str(h):>8}   {d.get('r2',float('nan')):>8.4f}   "
                  f"{d.get('r2_adj',float('nan')):>8.4f}   "
                  f"{d.get('r2_oos',float('nan')):>8.4f}")
        print(f"{'='*55}")

    print("  [r2]   using inline R² definitions")


# ─────────────────────────────────────────────────────────────────────────
# C.  Stationarity (ADF + fractional differencing)
# ─────────────────────────────────────────────────────────────────────────
try:
    from src.data.stationarity import (
        adf_test, kpss_test, test_feature_stationarity,
        fractional_difference, make_stationary_features,
        apply_stationarity_transform,
    )
    print("  [stat] loaded from src.data.stationarity")
except ImportError:
    def adf_test(series, p_threshold=0.05):
        from statsmodels.tsa.stattools import adfuller
        x = np.asarray(series, dtype=np.float64)
        x = x[~np.isnan(x)]
        res = adfuller(x, autolag="AIC")
        return {"statistic": float(res[0]), "p_value": float(res[1]),
                "is_stationary": bool(res[1] < p_threshold), "method": "ADF"}

    def kpss_test(series, p_threshold=0.05):
        from statsmodels.tsa.stattools import kpss
        x = np.asarray(series, dtype=np.float64)
        x = x[~np.isnan(x)]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            res = kpss(x, regression="c", nlags="auto")
        return {"statistic": float(res[0]), "p_value": float(res[1]),
                "is_stationary": bool(res[1] > p_threshold), "method": "KPSS"}

    def _ffd_weights(d, size, threshold=1e-3):
        w = [1.0]
        for k in range(1, size):
            wk = -w[-1] * (d - k + 1) / k
            if abs(wk) < threshold: break
            w.append(wk)
        return np.array(w[::-1], dtype=np.float64)

    def fractional_difference(x, d, threshold=1e-3, fill_nan=True):
        x = np.asarray(x, dtype=np.float64)
        squeeze = x.ndim == 1
        if squeeze: x = x[:, None]
        n, nf = x.shape
        w = _ffd_weights(d, n, threshold); win = len(w)
        out = np.full((n, nf), np.nan)
        for j in range(nf):
            col = x[:, j]
            for t in range(win - 1, n):
                seg = col[t - win + 1: t + 1]
                if not np.any(np.isnan(seg)):
                    out[t, j] = np.dot(w, seg)
        if not fill_nan: out = np.where(np.isnan(out), 0.0, out)
        return (out[:, 0] if squeeze else out).astype(np.float32)

    def test_feature_stationarity(X, feature_names=None, run_kpss=True,
                                   adf_p_threshold=0.05, verbose=True):
        import pandas as _pd
        X = np.asarray(X, dtype=np.float64)
        nf = X.shape[1]
        if feature_names is None: feature_names = [f"f{i}" for i in range(nf)]
        rows = []
        for j, name in enumerate(feature_names):
            r = {"feature": name}
            a = adf_test(X[:, j], adf_p_threshold)
            r["adf_stat"] = a["statistic"]; r["adf_pval"] = a["p_value"]; r["adf_stationary"] = a["is_stationary"]
            if run_kpss:
                try:
                    k = kpss_test(X[:, j])
                    r["kpss_stationary"] = k["is_stationary"]
                    r["verdict"] = "stationary" if a["is_stationary"] and k["is_stationary"] \
                        else ("non-stationary" if not a["is_stationary"] and not k["is_stationary"] else "ambiguous")
                except Exception: r["verdict"] = "stationary" if a["is_stationary"] else "non-stationary"
            else:
                r["verdict"] = "stationary" if a["is_stationary"] else "non-stationary"
            rows.append(r)
        df = _pd.DataFrame(rows)
        if verbose:
            print(f"  Non-stationary: {(df['verdict']=='non-stationary').sum()}/{nf}  "
                  f"Stationary: {(df['verdict']=='stationary').sum()}/{nf}  "
                  f"Ambiguous: {(df['verdict']=='ambiguous').sum()}/{nf}")
        return df

    def make_stationary_features(X, feature_names=None, method="diff",
                                  d_fixed=0.4, adf_p_threshold=0.05,
                                  ffd_threshold=1e-3, verbose=True):
        X = np.asarray(X, dtype=np.float64)
        n, nf = X.shape
        X_out = X.copy()
        d_per = [0.0] * nf
        idx_t = []
        if method == "diff":
            for j in range(nf):
                d = np.diff(X[:, j], prepend=X[0, j])
                d[0] = 0.0; X_out[:, j] = d; d_per[j] = 1.0; idx_t.append(j)
        elif method == "frac_diff":
            for j in range(nf):
                fd = fractional_difference(X[:, j], d_fixed, ffd_threshold)
                X_out[:, j] = np.where(np.isnan(fd), 0.0, fd); d_per[j] = d_fixed; idx_t.append(j)
        elif method == "auto":
            stat_df = test_feature_stationarity(X, feature_names, adf_p_threshold=adf_p_threshold, verbose=verbose)
            for j, row in stat_df.iterrows():
                if row["verdict"] == "non-stationary":
                    fd = fractional_difference(X[:, j], d_fixed, ffd_threshold)
                    X_out[:, j] = np.where(np.isnan(fd), 0.0, fd); d_per[j] = d_fixed; idx_t.append(j)
        X_out = np.where(np.isnan(X_out), 0.0, X_out)
        if verbose:
            print(f"  Stationarity transform: {len(set(idx_t))}/{nf} features transformed (method={method})")
        return X_out.astype(np.float32), {"method": method, "d_per_feature": d_per, "transformed_indices": idx_t}

    def apply_stationarity_transform(X, metadata, ffd_threshold=1e-3):
        X = np.asarray(X, dtype=np.float64)
        d_per = metadata["d_per_feature"]
        X_out = X.copy()
        for j, d in enumerate(d_per):
            if d == 0.0: continue
            elif d == 1.0:
                di = np.diff(X[:, j], prepend=X[0, j]); di[0] = 0.0; X_out[:, j] = di
            else:
                fd = fractional_difference(X[:, j], d, ffd_threshold)
                X_out[:, j] = np.where(np.isnan(fd), 0.0, fd)
        return np.where(np.isnan(X_out), 0.0, X_out).astype(np.float32)

    print("  [stat] using inline stationarity definitions")


# ─────────────────────────────────────────────────────────────────────────
# D.  SMOTE / class-imbalance diagnostics
# ─────────────────────────────────────────────────────────────────────────
try:
    from src.data.smote_preprocessing import (
        get_class_distribution, print_class_distribution,
        imbalance_ratio, apply_smote, apply_borderline_smote,
        apply_adasyn, apply_smote_tomek, RESAMPLE_METHODS,
    )
    print("  [smote] loaded from src.data.smote_preprocessing")
except ImportError:
    _CLASS_NAMES = {0: "DOWN", 1: "STATIONARY", 2: "UP"}

    def get_class_distribution(y, n_classes=3):
        y = np.asarray(y, dtype=np.int32).ravel()
        c = np.bincount(y, minlength=n_classes)
        return {int(i): int(c[i]) for i in range(n_classes)}

    def print_class_distribution(y, label="", n_classes=3):
        dist = get_class_distribution(y, n_classes)
        total = sum(dist.values())
        tag = f" [{label}]" if label else ""
        print(f"\nClass distribution{tag}  (total={total:,})")
        print("-" * 44)
        for cls, cnt in dist.items():
            name = _CLASS_NAMES.get(cls, f"cls{cls}")
            pct = 100.0 * cnt / max(total, 1)
            bar = "█" * int(pct / 2)
            print(f"  {cls} ({name:>10}): {cnt:>8,}  ({pct:5.1f}%)  {bar}")
        print("-" * 44)

    def imbalance_ratio(y, n_classes=3):
        d = get_class_distribution(y, n_classes)
        c = [v for v in d.values() if v > 0]
        return float(max(c) / min(c)) if c else 1.0

    def _smote_available():
        try: from imblearn.over_sampling import SMOTE; return True
        except ImportError: return False

    def apply_smote(X, y, sampling_strategy="auto", k_neighbors=5, random_state=42, verbose=True):
        if not _smote_available():
            raise ImportError("pip install imbalanced-learn")
        from imblearn.over_sampling import SMOTE
        if verbose: print_class_distribution(y, "Before SMOTE")
        sm = SMOTE(sampling_strategy=sampling_strategy, k_neighbors=k_neighbors, random_state=random_state)
        Xr, yr = sm.fit_resample(X, y)
        if verbose: print_class_distribution(yr, "After SMOTE")
        return Xr.astype(np.float32), yr.astype(np.int32)

    def apply_borderline_smote(X, y, sampling_strategy="auto", k_neighbors=5, random_state=42, verbose=True):
        if not _smote_available(): raise ImportError("pip install imbalanced-learn")
        from imblearn.over_sampling import BorderlineSMOTE
        if verbose: print_class_distribution(y, "Before Borderline-SMOTE")
        sm = BorderlineSMOTE(sampling_strategy=sampling_strategy, k_neighbors=k_neighbors, random_state=random_state)
        Xr, yr = sm.fit_resample(X, y)
        if verbose: print_class_distribution(yr, "After Borderline-SMOTE")
        return Xr.astype(np.float32), yr.astype(np.int32)

    def apply_adasyn(X, y, sampling_strategy="auto", n_neighbors=5, random_state=42, verbose=True):
        if not _smote_available(): raise ImportError("pip install imbalanced-learn")
        from imblearn.over_sampling import ADASYN
        if verbose: print_class_distribution(y, "Before ADASYN")
        sm = ADASYN(sampling_strategy=sampling_strategy, n_neighbors=n_neighbors, random_state=random_state)
        try:
            Xr, yr = sm.fit_resample(X, y)
        except ValueError:
            warnings.warn("ADASYN failed, falling back to SMOTE")
            return apply_smote(X, y, sampling_strategy=sampling_strategy, k_neighbors=n_neighbors,
                               random_state=random_state, verbose=False)
        if verbose: print_class_distribution(yr, "After ADASYN")
        return Xr.astype(np.float32), yr.astype(np.int32)

    def apply_smote_tomek(X, y, sampling_strategy="auto", random_state=42, verbose=True):
        if not _smote_available(): raise ImportError("pip install imbalanced-learn")
        from imblearn.combine import SMOTETomek
        if verbose: print_class_distribution(y, "Before SMOTE-Tomek")
        sm = SMOTETomek(sampling_strategy=sampling_strategy, random_state=random_state)
        Xr, yr = sm.fit_resample(X, y)
        if verbose: print_class_distribution(yr, "After SMOTE-Tomek")
        return Xr.astype(np.float32), yr.astype(np.int32)

    RESAMPLE_METHODS = {
        "smote": apply_smote,
        "borderline_smote": apply_borderline_smote,
        "adasyn": apply_adasyn,
        "smote_tomek": apply_smote_tomek,
    }
    print("  [smote] using inline SMOTE definitions")


# ── Global stationarity metadata (fitted on training data in Cell 4c) ─────
_STATIONARITY_META: Optional[dict] = None

print("\n✓ All accuracy-improvement modules loaded.")
print(f"  Normalization method : {DEEP_CONFIG.get('normalization_method','robust')}")
print(f"  SMOTE enabled        : {DEEP_CONFIG.get('enable_smote', False)}")
print(f"  Stationarity enabled : {DEEP_CONFIG.get('enable_stationarity', False)}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 5 — Checkpoint / Recovery System
# ═══════════════════════════════════════════════════════════════════════════
"""
The checkpoint system saves training state after EVERY epoch so that if
Colab crashes or times out, the next run can resume exactly where it left off.

State persisted per architecture:
  - Model weights (best so far)
  - Current epoch weights (for mid-epoch restart)
  - Epoch history (losses, f1 scores)
  - Best score / no_improve counter
  - Completed flag

All checkpoint files live in WEIGHTS_DIR (Google Drive in Colab).
"""
import json


def _ckpt_path(arch: str, suffix: str = "_stable_es") -> str:
    """Path for the JSON checkpoint file."""
    return os.path.join(WEIGHTS_DIR, f"{arch}{suffix}_checkpoint.json")


def _ckpt_weights_path(arch: str, label: str = "current", suffix: str = "_stable_es") -> str:
    """Path for a .pt weights file (current or best)."""
    return os.path.join(WEIGHTS_DIR, f"{arch}{suffix}_{label}_weights.pt")


def save_checkpoint(
    arch: str,
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    best_score: float,
    best_epoch: int,
    no_improve: int,
    epoch_history: list,
    best_metrics: dict,
    completed: bool = False,
    suffix: str = "_stable_es",
) -> None:
    """
    Persist model + training state after each epoch.
    Called inside the training loop — safe to interrupt during next epoch.
    """
    # 1. Save current model weights
    cur_w_path = _ckpt_weights_path(arch, "current", suffix)
    torch.save(model.state_dict(), cur_w_path)

    # 2. Save JSON metadata
    meta = {
        "arch":          arch,
        "epoch":         int(epoch),
        "best_score":    float(best_score),
        "best_epoch":    int(best_epoch),
        "no_improve":    int(no_improve),
        "completed":     bool(completed),
        "epoch_history": epoch_history,
        "best_metrics":  best_metrics,
        "timestamp":     pd.Timestamp.now().isoformat(),
    }
    ckpt_p = _ckpt_path(arch, suffix)
    # Atomic-ish write: write to temp then rename
    tmp_p = ckpt_p + ".tmp"
    with open(tmp_p, "w") as f:
        json.dump(meta, f, indent=2)
    os.replace(tmp_p, ckpt_p)


def save_best_weights(
    arch: str,
    model: nn.Module,
    suffix: str = "_stable_es",
) -> str:
    """Save the best model weights separately."""
    best_w_path = _ckpt_weights_path(arch, "best", suffix)
    torch.save(model.state_dict(), best_w_path)
    return best_w_path


def load_checkpoint(arch: str, suffix: str = "_stable_es") -> dict | None:
    """
    Load checkpoint metadata for `arch`. Returns None if no checkpoint exists.
    """
    ckpt_p = _ckpt_path(arch, suffix)
    if not os.path.exists(ckpt_p):
        return None
    try:
        with open(ckpt_p) as f:
            return json.load(f)
    except Exception as e:
        print(f"[ckpt] WARNING: could not read checkpoint for {arch}: {e}")
        return None


def is_arch_completed(arch: str, suffix: str = "_stable_es") -> bool:
    """Return True if this architecture finished training."""
    ckpt = load_checkpoint(arch, suffix)
    return ckpt is not None and bool(ckpt.get("completed", False))


def restore_weights_to_model(
    arch: str,
    model: nn.Module,
    label: str = "current",
    suffix: str = "_stable_es",
) -> bool:
    """
    Load saved weights into model (in-place). Returns True on success.
    label: "current" or "best"
    """
    path = _ckpt_weights_path(arch, label, suffix)
    if not os.path.exists(path):
        return False
    try:
        sd = torch.load(path, map_location="cpu")
        model.load_state_dict(sd)
        return True
    except Exception as e:
        print(f"[ckpt] WARNING: could not load {label} weights for {arch}: {e}")
        return False


def print_checkpoint_status(suffix: str = "_stable_es") -> None:
    """Print recovery status for all architectures in DEEP_CONFIG."""
    archs = DEEP_CONFIG.get("run_architectures", [])
    print("\n" + "═" * 60)
    print("  Checkpoint Status")
    print("═" * 60)
    for arch in archs:
        ckpt = load_checkpoint(arch, suffix)
        if ckpt is None:
            status = "NOT STARTED"
        elif ckpt.get("completed"):
            ep  = ckpt.get('epoch', '?')
            sc  = ckpt.get('best_score', 0.0)
            status = f"COMPLETED  epoch={ep}  best_f1={sc:.4f}"
        else:
            ep = ckpt.get('epoch', 0)
            sc = ckpt.get('best_score', 0.0)
            ni = ckpt.get('no_improve', 0)
            status = f"IN PROGRESS  last_epoch={ep}  best_f1={sc:.4f}  no_improve={ni}"
        print(f"  {arch:<30} {status}")
    print("═" * 60 + "\n")


print_checkpoint_status()
print("Checkpoint system ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 6 — Training Utilities
# ═══════════════════════════════════════════════════════════════════════════
import psutil


def deep_set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _avail_ram_gb() -> float:
    return psutil.virtual_memory().available / 1e9


def _deep_cleanup_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _deep_choose_subsample(cfg: dict) -> int:
    free = _avail_ram_gb()
    if free < 2.0:
        return int(cfg.get("critical_pressure_subsample", 16))
    if free < 4.0:
        return int(cfg.get("high_pressure_subsample", 8))
    return int(cfg.get("base_subsample", 4))


def _deep_choose_max_rows(cfg: dict, is_train: bool) -> int:
    key = "max_rows_per_day_train" if is_train else "max_rows_per_day_eval"
    default = 8000 if is_train else 10000
    return int(max(1, cfg.get(key, default)))


def _deep_seed_from_path(path: str, base_seed: int) -> int:
    digest = hashlib.md5(path.encode("utf-8")).hexdigest()
    return (int(digest[:8], 16) + int(base_seed)) % (2 ** 32 - 1)


def _deep_resolve_tickers(cfg: dict) -> list:
    if cfg.get("tickers"):
        return list(cfg["tickers"])
    data_dir = cfg["data_dir"]
    return sorted([
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
        and glob.glob(os.path.join(data_dir, d, "*.parquet"))
    ])


def _deep_collect_files_by_ticker(
    data_dir: str,
    tickers: list,
    max_files: int = 0,
) -> dict:
    files_by_ticker: Dict[str, List[str]] = {}
    for ticker in tickers:
        files = sorted(glob.glob(os.path.join(data_dir, ticker, "*.parquet")))
        if max_files > 0:
            files = files[:max_files]
        if files:
            files_by_ticker[ticker] = files
    return files_by_ticker


def _deep_split_train_eval_files(
    files_by_ticker: dict,
    train_frac: float = 0.8,
) -> Tuple[List[Tuple[str, str]], List[Tuple[str, str]]]:
    train_files: List[Tuple[str, str]] = []
    eval_files:  List[Tuple[str, str]] = []
    for ticker, files in files_by_ticker.items():
        n = len(files)
        n_train = max(1, min(n - 1, int(np.floor(n * train_frac)))) if n > 1 else 1
        for i, p in enumerate(files):
            (train_files if i < n_train else eval_files).append((ticker, p))
    return train_files, eval_files


def _deep_update_confusion(confusion: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray) -> None:
    mask = (y_true >= 0) & (y_true < 3) & (y_pred >= 0) & (y_pred < 3)
    yt = y_true[mask].astype(np.int64)
    yp = y_pred[mask].astype(np.int64)
    np.add.at(confusion, (yt, yp), 1)


def _deep_metrics_from_confusion(confusion: np.ndarray, h: int) -> dict:
    metrics: Dict[str, float] = {}
    n_cls = confusion.shape[0]
    total = float(confusion.sum())
    metrics[f"h{h}_accuracy"] = float(np.trace(confusion)) / max(total, 1.0)
    f1s = []
    for c in range(n_cls):
        tp = float(confusion[c, c])
        fp = float(confusion[:, c].sum()) - tp
        fn = float(confusion[c, :].sum()) - tp
        prec = tp / max(tp + fp, 1e-9)
        rec  = tp / max(tp + fn, 1e-9)
        f1   = 2.0 * prec * rec / max(prec + rec, 1e-9)
        f1s.append(f1)
        metrics[f"h{h}_f1_c{c}"] = f1
    metrics[f"h{h}_f1_macro"]    = float(np.mean(f1s))
    metrics[f"h{h}_f1_weighted"] = float(
        sum(f1s[c] * float(confusion[c, :].sum()) for c in range(n_cls)) / max(total, 1.0)
    )
    return metrics


def _mean_macro_f1(metrics: dict, horizons: list) -> float:
    vals = [float(metrics.get(f"h{h}_f1_macro", 0.0)) for h in horizons]
    return float(np.mean(vals)) if vals else 0.0


print("Utilities ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 4c — Data Diagnostics  (stationarity + class imbalance)
#
# Run this cell BEFORE training to understand your data and validate config.
# It samples a few training files, runs stationarity tests, shows class
# distributions, and optionally fits the stationarity transform.
#
# Set DIAG_N_FILES to control how many files are sampled (keep ≤5 for speed).
# ═══════════════════════════════════════════════════════════════════════════

DIAG_N_FILES = 3   # number of parquet files to sample for diagnostics

_tickers_diag = _deep_resolve_tickers(DEEP_CONFIG)
_files_diag   = _deep_collect_files_by_ticker(
    DEEP_CONFIG["data_dir"], _tickers_diag,
    int(DEEP_CONFIG.get("max_files_per_ticker", 0))
)

# Flatten and take first DIAG_N_FILES
_sample_files = []
for ticker, paths in _files_diag.items():
    for p in paths:
        _sample_files.append((ticker, p))
        if len(_sample_files) >= DIAG_N_FILES:
            break
    if len(_sample_files) >= DIAG_N_FILES:
        break

print(f"Sampling {len(_sample_files)} file(s) for diagnostics ...\n")

# ── Aggregate raw arrays and labels ──────────────────────────────────────
_all_raw    = []
_all_labels = {h: [] for h in DEEP_CONFIG["horizons"]}

for _ticker, _path in _sample_files:
    try:
        _df   = pd.read_parquet(_path, columns=DEEP_RAW_LOB_10_COLS)
        _raw  = _df.to_numpy(dtype=np.float32, copy=False)
        _raw  = np.nan_to_num(_raw, nan=0.0, posinf=0.0, neginf=0.0)
        _ydf  = make_fixed_threshold_classification_labels(
            _df, horizons=DEEP_CONFIG["horizons"],
            alpha=float(DEEP_CONFIG["alpha"]), use_smoothing=True
        )
        _max_h = max(DEEP_CONFIG["horizons"])
        _valid = ~_ydf.iloc[:len(_df)-_max_h].isna().any(axis=1)
        _raw_v = _raw[:len(_df)-_max_h][_valid.values]
        _all_raw.append(_raw_v[:2000])  # cap per-file rows
        for h in DEEP_CONFIG["horizons"]:
            _all_labels[h].append(_ydf[f"label_{h}"].values[:len(_df)-_max_h][_valid.values][:2000])
        print(f"  Loaded {_ticker}/{os.path.basename(_path)}  rows={_valid.sum()}")
        del _df, _raw, _ydf, _valid, _raw_v
        gc.collect()
    except Exception as _e:
        print(f"  SKIP {_path}: {_e}")

if not _all_raw:
    print("No files loaded — check DATA_DIR and tickers.")
else:
    _X_sample = np.vstack(_all_raw)
    print(f"\nSample matrix: {_X_sample.shape[0]:,} rows × {_X_sample.shape[1]} features")

    # ── Class distribution ────────────────────────────────────────────────
    print("\n" + "═"*55)
    print("  Class Distribution")
    print("═"*55)
    for h in DEEP_CONFIG["horizons"]:
        _y_h = np.concatenate(_all_labels[h]).astype(np.int32)
        _y_h = _y_h[(_y_h >= 0) & (_y_h <= 2)]
        _ir  = imbalance_ratio(_y_h)
        print(f"\n  Horizon k={h}")
        print_class_distribution(_y_h, label=f"h{h}")
        print(f"  Imbalance ratio: {_ir:.2f}x  ", end="")
        if _ir > 5:
            print("⚠ HIGH — consider enable_smote=True or cb_focal loss")
        elif _ir > 3:
            print("⚠ MODERATE — cb_focal loss recommended")
        else:
            print("✓ acceptable")

    # ── Stationarity test ─────────────────────────────────────────────────
    print("\n" + "═"*55)
    print("  Stationarity Test  (ADF + KPSS on sample)")
    print("═"*55)
    try:
        _stat_df = test_feature_stationarity(
            _X_sample[:1000],                  # use first 1000 rows for speed
            feature_names=DEEP_RAW_LOB_10_COLS,
            run_kpss=True,
            adf_p_threshold=0.05,
            verbose=True,
        )
        _n_nonstat = (_stat_df["verdict"] == "non-stationary").sum()
        print(f"\n  → {_n_nonstat}/{len(DEEP_RAW_LOB_10_COLS)} features are non-stationary")
        if _n_nonstat > 0 and not DEEP_CONFIG.get("enable_stationarity"):
            print("  ℹ  Consider setting enable_stationarity=True in Cell 4 config to")
            print("     apply fractional differencing before model input.")
    except Exception as _se:
        print(f"  Stationarity test skipped: {_se}")
        _stat_df = None

    # ── Fit stationarity transform on sample (if enabled) ─────────────────
    if DEEP_CONFIG.get("enable_stationarity") and _X_sample.shape[0] >= 50:
        print("\nFitting stationarity transform ...")
        _, _STATIONARITY_META = make_stationary_features(
            _X_sample,
            feature_names=DEEP_RAW_LOB_10_COLS,
            method=str(DEEP_CONFIG.get("stationarity_method", "auto")),
            d_fixed=float(DEEP_CONFIG.get("stationarity_d_fixed", 0.4)),
            verbose=True,
        )
        print("✓ Stationarity transform fitted. Will be applied in dataset builder.")
    else:
        _STATIONARITY_META = None
        if DEEP_CONFIG.get("enable_stationarity"):
            print("  Not enough data to fit stationarity transform.")

    # ── Normalization preview ─────────────────────────────────────────────
    print("\n" + "═"*55)
    print(f"  Normalization Method: {DEEP_CONFIG.get('normalization_method','robust')}")
    print("═"*55)
    _norm_method = str(DEEP_CONFIG.get("normalization_method", "robust"))
    try:
        _scaler_cls = {
            "zscore": ZScoreScaler, "minmax": MinMaxScaler, "robust": RobustScaler,
            "maxabs": MaxAbsScaler, "decimal": DecimalScaler,
            "quantile": lambda: QuantileScaler("uniform"),
            "quantile_normal": lambda: QuantileScaler("normal"),
            "power": PowerTransformer,
        }.get(_norm_method, RobustScaler)
        _s_demo = (_scaler_cls() if isinstance(_scaler_cls, type) else _scaler_cls())
        _X_normed = _s_demo.fit_transform(_X_sample.astype(np.float64)).astype(np.float32)
        print(f"  Sample range after {_norm_method}: [{_X_normed.min():.3f}, {_X_normed.max():.3f}]")
        print(f"  Mean={_X_normed.mean():.4f}  Std={_X_normed.std():.4f}")
        del _X_normed, _s_demo
    except Exception as _ne:
        print(f"  Normalization preview failed: {_ne}")

    del _X_sample
    gc.collect()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 7 — Dataset  (Stable Day-Window + configurable normalization + SMOTE)
# ═══════════════════════════════════════════════════════════════════════════
from dataclasses import dataclass


@dataclass
class DayStats:
    ticker: str
    file_name: str
    rows_kept: int
    step: int
    max_rows: int


def _apply_day_normalization(raw: np.ndarray, method: str) -> np.ndarray:
    """
    Normalise a single day's raw feature matrix.

    Per-day normalization is important: different days can have very different
    price scales, so a globally-fitted scaler may not generalise well.
    We fit on the current day and apply within that day only.
    """
    raw = raw.astype(np.float64)
    try:
        if method == "zscore":
            mean = raw.mean(0); std = np.where(raw.std(0) < 1e-8, 1.0, raw.std(0))
            return ((raw - mean) / std).astype(np.float32)
        elif method == "minmax":
            mn = raw.min(0); mx = raw.max(0); r = np.where(mx - mn < 1e-8, 1.0, mx - mn)
            return ((raw - mn) / r).astype(np.float32)
        elif method == "robust":
            med = np.median(raw, 0)
            q75, q25 = np.percentile(raw, [75, 25], 0)
            iqr = np.where(q75 - q25 < 1e-8, 1.0, q75 - q25)
            return ((raw - med) / iqr).astype(np.float32)
        elif method == "maxabs":
            ma = np.where(np.abs(raw).max(0) < 1e-8, 1.0, np.abs(raw).max(0))
            return (raw / ma).astype(np.float32)
        elif method == "decimal":
            ma = np.abs(raw).max(0)
            scale = np.power(10, np.ceil(np.log10(np.where(ma < 1e-8, 1.0, ma))))
            return (raw / scale).astype(np.float32)
        elif method in ("quantile", "quantile_normal"):
            try:
                from sklearn.preprocessing import QuantileTransformer
                dist = "normal" if method == "quantile_normal" else "uniform"
                qt = QuantileTransformer(output_distribution=dist,
                                         n_quantiles=min(1000, len(raw)), random_state=42)
                return qt.fit_transform(raw).astype(np.float32)
            except ImportError:
                pass  # fall through to zscore
        elif method == "power":
            try:
                from sklearn.preprocessing import PowerTransformer
                pt = PowerTransformer(method="yeo-johnson", standardize=True)
                return pt.fit_transform(raw).astype(np.float32)
            except ImportError:
                pass  # fall through to zscore
    except Exception:
        pass
    # Default fallback: zscore
    mean = raw.mean(0); std = np.where(raw.std(0) < 1e-8, 1.0, raw.std(0))
    return ((raw - mean) / std).astype(np.float32)


class StableDayWindowDataset(Dataset):
    """
    Per-day normalized sliding-window dataset.

    Supports multiple normalization methods via DEEP_CONFIG['normalization_method'].
    The normalization is fitted and applied per day to prevent cross-day leakage.
    """

    def __init__(self, raw: np.ndarray, starts: np.ndarray, labels: np.ndarray,
                 seq_len: int, norm_method: str = "robust"):
        raw  = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)
        # Apply normalization before storing (once per day, not per sample)
        raw_normed = _apply_day_normalization(raw, norm_method)
        self.raw    = raw_normed
        self.starts = starts.astype(np.int64, copy=False)
        self.labels = labels.astype(np.int64, copy=False)
        self.seq_len = int(seq_len)

    def __len__(self) -> int:
        return int(self.starts.size)

    def __getitem__(self, idx: int):
        s = int(self.starts[idx])
        x = self.raw[s: s + self.seq_len].copy()
        x = np.clip(np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0), -10.0, 10.0)
        y = self.labels[idx]
        return torch.from_numpy(x.astype(np.float32, copy=False)), torch.from_numpy(y)


def _apply_smote_to_day(
    raw: np.ndarray,
    starts: np.ndarray,
    y_day: np.ndarray,
    cfg: dict,
    parquet_path: str,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Apply SMOTE to the flattened window features for a single day.
    Returns (raw_aug, starts_aug, y_day_aug) — synthetic samples appended.

    SMOTE is applied to the FLATTENED window vectors (seq_len × n_features)
    so the resampler sees the full temporal context per sample.
    NOTE: This is memory-intensive. Keep seq_len × batch small in Colab.
    """
    seq_len     = int(cfg["seq_len"])
    smote_meth  = str(cfg.get("smote_method", "smote"))
    k_neighbors = int(cfg.get("smote_k", 5))
    min_per_cls = int(cfg.get("smote_min_per_class", 10))
    seed        = _deep_seed_from_path(parquet_path, int(cfg.get("seed", 42)))

    # Use first horizon's label for SMOTE
    y_h = y_day[:, 0].astype(np.int32)
    counts = np.bincount(y_h, minlength=3)
    if int(counts.min()) < min_per_cls:
        return raw, starts, y_day   # not enough samples for SMOTE

    # Build flat feature matrix: N × (seq_len * n_features)
    n_samples = len(starts)
    n_features = raw.shape[1]
    X_flat = np.zeros((n_samples, seq_len * n_features), dtype=np.float32)
    for i, s in enumerate(starts):
        X_flat[i] = raw[s: s + seq_len].ravel()

    try:
        fn = RESAMPLE_METHODS.get(smote_meth, apply_smote)
        X_res, y_res = fn(X_flat, y_h, k_neighbors=k_neighbors,
                          random_state=seed, verbose=False)
    except Exception as _e:
        warnings.warn(f"SMOTE failed ({_e}), using original data")
        return raw, starts, y_day

    n_synth  = len(X_res) - n_samples
    if n_synth <= 0:
        return raw, starts, y_day

    # For synthetic samples: unflatten to (n_synth, seq_len, n_features),
    # append to raw, create virtual start indices
    X_synth = X_res[n_samples:].reshape(n_synth, seq_len, n_features)
    n_raw   = len(raw)
    raw_ext = np.vstack([raw, X_synth.reshape(-1, n_features)])
    synth_starts = np.array([n_raw + i * seq_len for i in range(n_synth)], dtype=np.int64)
    starts_ext   = np.concatenate([starts, synth_starts])

    # Replicate labels from first horizon for all horizons
    y_synth = np.tile(y_res[n_samples:, None], (1, y_day.shape[1])).astype(np.int64)
    y_ext   = np.vstack([y_day, y_synth])

    return raw_ext, starts_ext, y_ext


def _deep_build_day_dataset(
    parquet_path: str,
    cfg: dict,
    is_train: bool,
) -> Tuple["StableDayWindowDataset | None", "DayStats | None"]:
    horizons = list(cfg["horizons"])
    seq_len  = int(cfg["seq_len"])
    alpha    = float(cfg["alpha"])
    step     = _deep_choose_subsample(cfg)
    max_rows = _deep_choose_max_rows(cfg, is_train=is_train)
    norm_method = str(cfg.get("normalization_method", "robust"))

    try:
        df = pd.read_parquet(parquet_path, columns=DEEP_RAW_LOB_10_COLS)
    except Exception as e:
        print(f"  [dataset] Failed to read {parquet_path}: {e}")
        return None, None

    raw    = np.ascontiguousarray(df.to_numpy(dtype=np.float32, copy=False))
    raw    = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)
    y_full = make_fixed_threshold_classification_labels(
        df, horizons=horizons, alpha=alpha, use_smoothing=True
    ).to_numpy(dtype=np.float32, copy=False)

    # Apply stationarity transform if fitted
    if cfg.get("enable_stationarity") and "_STATIONARITY_META" in globals() and _STATIONARITY_META is not None:
        try:
            raw = apply_stationarity_transform(raw, _STATIONARITY_META)
        except Exception as _se:
            warnings.warn(f"Stationarity transform failed: {_se}")

    max_h     = int(max(horizons))
    valid_end = len(df) - max_h
    del df

    if valid_end <= seq_len:
        gc.collect()
        return None, None

    labels     = y_full[seq_len - 1: valid_end]
    valid_mask = ~np.isnan(labels).any(axis=1)
    starts     = np.flatnonzero(valid_mask).astype(np.int64, copy=False)
    if step > 1:
        starts = starts[::step]
    if max_rows > 0:
        starts = starts[:max_rows]
    if starts.size == 0:
        gc.collect()
        return None, None

    y_day      = labels[starts].astype(np.int64, copy=False)
    class_mask = ((y_day >= 0) & (y_day <= 2)).all(axis=1)
    starts     = starts[class_mask]
    y_day      = y_day[class_mask]

    if starts.size == 0:
        gc.collect()
        return None, None

    # Apply SMOTE on training data only
    if is_train and cfg.get("enable_smote", False):
        try:
            raw, starts, y_day = _apply_smote_to_day(raw, starts, y_day, cfg, parquet_path)
        except Exception as _se:
            warnings.warn(f"SMOTE failed for {parquet_path}: {_se}")

    ds = StableDayWindowDataset(raw, starts, y_day, seq_len, norm_method=norm_method)
    stats = DayStats(
        ticker=os.path.basename(os.path.dirname(parquet_path)),
        file_name=os.path.basename(parquet_path),
        rows_kept=int(starts.size),
        step=step,
        max_rows=max_rows,
    )
    del raw, y_full, labels, valid_mask, y_day, class_mask
    gc.collect()
    return ds, stats


def _deep_make_loader(dataset: Dataset, cfg: dict, is_train: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=int(cfg.get("batch_size", 256)),
        shuffle=is_train,
        num_workers=int(cfg.get("num_workers", 0)),
        pin_memory=(DEEP_DEVICE.type == "cuda"),
        drop_last=is_train,
    )


print("Dataset classes ready.")
print(f"  Normalization: {DEEP_CONFIG.get('normalization_method', 'robust')}")
print(f"  SMOTE:         {DEEP_CONFIG.get('enable_smote', False)}")
print(f"  Stationarity:  {DEEP_CONFIG.get('enable_stationarity', False)}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 8 — Loss Functions (Class-Balanced Focal Loss)
# ═══════════════════════════════════════════════════════════════════════════

def _deep_class_weights_cb(labels: np.ndarray, cfg: dict) -> List[np.ndarray]:
    """Effective-Number-of-Samples class weights (Cui et al. CVPR 2019)."""
    beta  = float(cfg.get("cb_beta", 0.999))
    min_w = float(cfg.get("cb_min_w", 0.5))
    max_w = float(cfg.get("cb_max_w", 3.0))
    eps   = float(cfg.get("cb_eps", 1.0))
    weights: List[np.ndarray] = []
    for i in range(labels.shape[1]):
        y      = labels[:, i]
        counts = np.bincount(y.astype(np.int64, copy=False), minlength=3).astype(np.float64) + eps
        eff    = 1.0 - np.power(beta, counts)
        w      = (1.0 - beta) / np.maximum(eff, 1e-12)
        w      = w / np.mean(w)
        w      = np.clip(w, min_w, max_w)
        weights.append(w.astype(np.float32))
    return weights


def _deep_focal_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    weight: Optional[torch.Tensor],
    gamma: float = 2.0,
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    ce   = F.cross_entropy(logits, targets, weight=weight, reduction="none", label_smoothing=label_smoothing)
    p_t  = torch.exp(-ce)
    loss = ((1.0 - p_t) ** gamma) * ce
    return loss.mean()


def _deep_multihorizon_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    cfg: dict,
    weight_tensors: Optional[List[torch.Tensor]] = None,
) -> torch.Tensor:
    loss_mode       = str(cfg.get("loss_mode", "ce"))
    focal_gamma     = float(cfg.get("focal_gamma", 2.0))
    label_smoothing = float(cfg.get("label_smoothing", 0.0))
    losses = []
    for i in range(logits.shape[1]):
        w = None if weight_tensors is None else weight_tensors[i]
        if loss_mode == "cb_focal":
            l_i = _deep_focal_loss(logits[:, i, :], targets[:, i], w, focal_gamma, label_smoothing)
        else:
            l_i = F.cross_entropy(logits[:, i, :], targets[:, i], weight=w, label_smoothing=label_smoothing)
        losses.append(l_i)
    return torch.stack(losses).mean()


print("Loss functions ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 9 — Model Definitions
# ═══════════════════════════════════════════════════════════════════════════

# ── 1. Dilated CNN + Masked Transformer ──────────────────────────────────

class CausalDilatedConv1d(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, dilation: int, dropout: float = 0.1):
        super().__init__()
        self.pad  = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, dilation=dilation, padding=self.pad)
        self.bn   = nn.BatchNorm1d(out_ch)
        self.drop = nn.Dropout(dropout)
        self.res  = nn.Identity() if in_ch == out_ch else nn.Conv1d(in_ch, out_ch, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.conv(x)
        if self.pad > 0:
            y = y[..., :-self.pad]
        y = self.drop(F.gelu(self.bn(y)))
        r = self.res(x)
        if r.shape[-1] != y.shape[-1]:
            r = r[..., -y.shape[-1]:]
        return y + r


class DilatedMaskedTransformer(nn.Module):
    def __init__(self, input_dim: int, horizon_count: int, num_classes: int = 3,
                 d_model: int = 96, n_heads: int = 4, n_layers: int = 2,
                 dropout: float = 0.15, max_len: int = 1024):
        super().__init__()
        self.horizon_count = horizon_count
        self.num_classes   = num_classes
        self.input_proj    = nn.Conv1d(input_dim, d_model, kernel_size=1)
        self.tcn = nn.ModuleList([
            CausalDilatedConv1d(d_model, d_model, kernel_size=3, dilation=1, dropout=dropout),
            CausalDilatedConv1d(d_model, d_model, kernel_size=3, dilation=2, dropout=dropout),
            CausalDilatedConv1d(d_model, d_model, kernel_size=3, dilation=4, dropout=dropout),
        ])
        self.pos_emb = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, horizon_count * num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz, seq_len, _ = x.shape
        z = self.input_proj(x.transpose(1, 2))
        for block in self.tcn:
            z = block(z)
        z = z.transpose(1, 2)
        pos = self.pos_emb[:, :seq_len] if self.pos_emb.size(1) >= seq_len else \
            F.interpolate(self.pos_emb.transpose(1, 2), size=seq_len, mode="linear", align_corners=False).transpose(1, 2)
        z += pos
        attn_mask = torch.triu(torch.full((seq_len, seq_len), float("-inf"), device=z.device), diagonal=1)
        z = self.norm(self.encoder(z, mask=attn_mask)[:, -1, :])
        return self.head(z).view(bsz, self.horizon_count, self.num_classes)


# ── 2. Hybrid CNN + Inception + LSTM ─────────────────────────────────────

class InceptionBlock1D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        b1 = out_ch // 4; b2 = out_ch // 4; b3 = out_ch // 4; b4 = out_ch - (b1 + b2 + b3)
        self.branch1 = nn.Sequential(nn.Conv1d(in_ch, b1, 1), nn.BatchNorm1d(b1), nn.GELU())
        self.branch2 = nn.Sequential(nn.Conv1d(in_ch, b2, 1), nn.BatchNorm1d(b2), nn.GELU(),
                                     nn.Conv1d(b2, b2, 3, padding=1), nn.BatchNorm1d(b2), nn.GELU())
        self.branch3 = nn.Sequential(nn.Conv1d(in_ch, b3, 1), nn.BatchNorm1d(b3), nn.GELU(),
                                     nn.Conv1d(b3, b3, 5, padding=2), nn.BatchNorm1d(b3), nn.GELU())
        self.branch4 = nn.Sequential(nn.MaxPool1d(3, stride=1, padding=1),
                                     nn.Conv1d(in_ch, b4, 1), nn.BatchNorm1d(b4), nn.GELU())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)


class HybridCNNInceptionLSTM(nn.Module):
    def __init__(self, input_dim: int, horizon_count: int, num_classes: int = 3,
                 channels: int = 96, lstm_hidden: int = 128, lstm_layers: int = 2, dropout: float = 0.15):
        super().__init__()
        self.horizon_count = horizon_count
        self.num_classes   = num_classes
        self.stem = nn.Sequential(
            nn.Conv1d(input_dim, channels, kernel_size=7, padding=3),
            nn.BatchNorm1d(channels), nn.GELU(), nn.Dropout(dropout),
        )
        self.inception = InceptionBlock1D(channels, channels)
        self.pool      = nn.AdaptiveAvgPool1d(32)
        self.lstm      = nn.LSTM(channels, lstm_hidden, num_layers=lstm_layers, batch_first=True,
                                 dropout=dropout if lstm_layers > 1 else 0.0, bidirectional=False)
        self.drop      = nn.Dropout(dropout)
        self.head      = nn.Linear(lstm_hidden, horizon_count * num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz = x.size(0)
        z = self.stem(x.transpose(1, 2))
        z = self.inception(z)
        z = self.pool(z).transpose(1, 2)
        _, (h, _) = self.lstm(z)
        z = self.drop(h[-1])
        return self.head(z).view(bsz, self.horizon_count, self.num_classes)


# ── 3. Seq2Seq + Attention (DeepLOB-style) ───────────────────────────────

class Seq2SeqAttentionModel(nn.Module):
    def __init__(self, input_dim: int, horizon_count: int, num_classes: int = 3,
                 enc_hidden: int = 64, dec_hidden: int = 64, enc_layers: int = 2,
                 attn_dim: int = 64, dropout: float = 0.3):
        super().__init__()
        self.horizon_count = horizon_count
        self.num_classes   = num_classes
        self.encoder = nn.LSTM(input_dim, enc_hidden, num_layers=enc_layers, batch_first=True,
                               dropout=dropout if enc_layers > 1 else 0.0, bidirectional=True)
        enc_out_dim = enc_hidden * 2
        self.attn_w  = nn.Linear(enc_out_dim, attn_dim, bias=False)
        self.attn_v  = nn.Linear(attn_dim, 1, bias=False)
        self.decoder = nn.LSTM(enc_out_dim, dec_hidden, num_layers=1, batch_first=True)
        self.drop    = nn.Dropout(dropout)
        self.head    = nn.Linear(dec_hidden, horizon_count * num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz = x.size(0)
        enc_out, _ = self.encoder(x)                       # (B, T, 2*H)
        scores     = self.attn_v(torch.tanh(self.attn_w(enc_out))).squeeze(-1)
        weights    = torch.softmax(scores, dim=1).unsqueeze(-1)
        context    = (enc_out * weights).sum(dim=1, keepdim=True)  # (B, 1, 2*H)
        _, (h, _)  = self.decoder(context)
        z          = self.drop(h[-1])
        return self.head(z).view(bsz, self.horizon_count, self.num_classes)


# ── Model factory ─────────────────────────────────────────────────────────

def build_deep_model(arch: str, input_dim: int, horizon_count: int, num_classes: int = 3) -> nn.Module:
    if arch == "dilated_transformer":
        return DilatedMaskedTransformer(input_dim, horizon_count, num_classes)
    elif arch == "cnn_inception_lstm":
        return HybridCNNInceptionLSTM(input_dim, horizon_count, num_classes)
    elif arch == "seq2seq_attn":
        return Seq2SeqAttentionModel(input_dim, horizon_count, num_classes)
    else:
        raise ValueError(f"Unknown architecture: '{arch}'. "
                         f"Choose from: dilated_transformer, cnn_inception_lstm, seq2seq_attn")


# Sanity check
with torch.no_grad():
    for _arch in DEEP_CONFIG["run_architectures"]:
        _m = build_deep_model(_arch, len(DEEP_RAW_LOB_10_COLS), len(DEEP_CONFIG["horizons"]))
        _x = torch.randn(2, DEEP_CONFIG["seq_len"], len(DEEP_RAW_LOB_10_COLS))
        _y = _m(_x)
        assert _y.shape == (2, len(DEEP_CONFIG["horizons"]), 3), f"{_arch}: unexpected shape {_y.shape}"
        n_params = sum(p.numel() for p in _m.parameters())
        print(f"  {_arch:<30} params={n_params:,}  output={tuple(_y.shape)}")
        del _m, _x, _y

print("Models OK.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 10 — Core Training & Evaluation Loops
# ═══════════════════════════════════════════════════════════════════════════

def _train_epoch(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: "torch.cuda.amp.GradScaler",
    cfg: dict,
    train_files: List[Tuple[str, str]],
    horizons: List[int],
    epoch_idx: int,
    total_epochs: int,
    amp_enabled: bool,
    arch: str,
) -> dict:
    model.train()
    grad_clip       = float(cfg.get("grad_clip", 1.0))
    label_smoothing = float(cfg.get("label_smoothing", 0.0))

    day_losses: List[float] = []
    bad_batches  = 0
    fitted_days  = 0
    train_rows   = {h: 0 for h in horizons}
    train_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}

    for file_idx, (ticker, path) in enumerate(train_files, start=1):
        ds, stats = _deep_build_day_dataset(path, cfg, is_train=True)
        if ds is None:
            continue

        # Compute class weights from this day's labels
        weight_tensors = None
        if cfg.get("loss_mode", "ce") == "cb_focal":
            cb_ws = _deep_class_weights_cb(ds.labels, cfg)
            weight_tensors = [torch.tensor(w, dtype=torch.float32, device=DEEP_DEVICE) for w in cb_ws]

        loader = _deep_make_loader(ds, cfg, is_train=True)
        file_losses: List[float] = []

        for xb, yb in loader:
            xb = xb.to(DEEP_DEVICE, non_blocking=True)
            xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0).clamp(-10.0, 10.0)
            yb = yb.to(DEEP_DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=amp_enabled):
                logits = model(xb)
                logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)
                loss   = _deep_multihorizon_loss(logits, yb, cfg, weight_tensors)

            if not torch.isfinite(loss):
                bad_batches += 1
                del xb, yb, logits, loss
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()

            file_losses.append(loss.item())
            batch_n = int(yb.shape[0])
            for i, h in enumerate(horizons):
                train_rows[h]   += batch_n
                train_counts[h] += np.bincount(yb[:, i].detach().cpu().numpy(), minlength=3)

            del xb, yb, logits, loss

        if file_losses:
            day_losses.append(float(np.mean(file_losses)))
            fitted_days += 1

        print(
            f"[{arch}][ep {epoch_idx}/{total_epochs}][train {file_idx}/{len(train_files)}] "
            f"{ticker}/{stats.file_name}  rows={stats.rows_kept}  sub={stats.step}  "
            f"avg_loss={day_losses[-1]:.4f}  ram={_avail_ram_gb():.1f}GB"
        )

        del loader, ds
        _deep_cleanup_cuda()

    return {
        "avg_train_loss":   float(np.mean(day_losses)) if day_losses else float("nan"),
        "fitted_days":       int(fitted_days),
        "bad_batches":       int(bad_batches),
        "train_rows_seen":   train_rows,
        "train_class_counts": train_counts,
    }


def _eval_epoch(
    model: nn.Module,
    cfg: dict,
    eval_files: List[Tuple[str, str]],
    horizons: List[int],
    arch: str,
) -> dict:
    confusion    = {h: np.zeros((3, 3), dtype=np.int64) for h in horizons}
    eval_rows    = {h: 0 for h in horizons}
    eval_counts  = {h: np.zeros(3, dtype=np.int64) for h in horizons}

    model.eval()
    with torch.no_grad():
        for file_idx, (ticker, path) in enumerate(eval_files, start=1):
            ds, stats = _deep_build_day_dataset(path, cfg, is_train=False)
            if ds is None:
                continue

            loader = _deep_make_loader(ds, cfg, is_train=False)
            for xb, yb in loader:
                xb    = torch.nan_to_num(xb.to(DEEP_DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0).clamp(-10.0, 10.0)
                logits = torch.nan_to_num(model(xb), nan=0.0, posinf=0.0, neginf=0.0)
                preds  = logits.argmax(dim=2).detach().cpu().numpy().astype(np.int64)
                y_true = yb.numpy().astype(np.int64)
                for i, h in enumerate(horizons):
                    _deep_update_confusion(confusion[h], y_true[:, i], preds[:, i])
                    eval_rows[h]   += int(y_true.shape[0])
                    eval_counts[h] += np.bincount(y_true[:, i], minlength=3)
                del xb, logits, preds, y_true

            print(f"[{arch}][eval {file_idx}/{len(eval_files)}] {ticker}/{stats.file_name}  rows={stats.rows_kept}")
            del loader, ds
            _deep_cleanup_cuda()

    metrics: Dict[str, float] = {}
    for h in horizons:
        metrics.update(_deep_metrics_from_confusion(confusion[h], h))

    return {"metrics": metrics, "confusion": confusion, "eval_rows_seen": eval_rows, "eval_class_counts": eval_counts}


print("Training/eval loops ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 11 — Per-Architecture Training with Full Checkpoint Recovery
# ═══════════════════════════════════════════════════════════════════════════

def train_one_architecture(
    arch: str,
    cfg: dict,
    max_epochs: int = 10,
    patience: int = 3,
    min_delta: float = 1e-4,
    force_restart: bool = False,
) -> dict:
    """
    Train one architecture with per-epoch checkpoint recovery.

    On restart:
    - If the architecture is already marked COMPLETED, return the saved result.
    - If partially trained, load the last saved weights + state and resume
      from the next epoch.
    - If force_restart=True, ignore any existing checkpoint and start fresh.
    """
    suffix   = str(cfg.get("result_suffix", "_stable_es"))
    horizons = list(cfg["horizons"])
    amp_enabled = bool(cfg.get("amp", False)) and DEEP_DEVICE.type == "cuda"

    # ── Recover: skip if already completed ──────────────────────────────
    if not force_restart and is_arch_completed(arch, suffix):
        ckpt = load_checkpoint(arch, suffix)
        print(f"[{arch}] Already COMPLETED (epoch={ckpt['epoch']}  best_f1={ckpt['best_score']:.4f}). Skipping.")
        # Load saved run_meta from results file
        results_path = os.path.join(cfg["results_dir"], f"{arch}_results_day_streaming{suffix}.json")
        if os.path.exists(results_path):
            with open(results_path) as f:
                return json.load(f)
        return {"architecture": arch, "completed": True, "skipped": True}

    # ── File discovery ───────────────────────────────────────────────────
    tickers_list = _deep_resolve_tickers(cfg)
    if not tickers_list:
        raise FileNotFoundError(f"No tickers found in {cfg['data_dir']}")

    files_by_ticker = _deep_collect_files_by_ticker(
        cfg["data_dir"], tickers_list, int(cfg.get("max_files_per_ticker", 0))
    )
    if not files_by_ticker:
        raise FileNotFoundError("No parquet files found for selected tickers.")

    train_files, eval_files = _deep_split_train_eval_files(
        files_by_ticker, float(cfg.get("train_file_fraction", 0.8))
    )

    print(f"\n{'═'*80}")
    print(f"  Training: {arch}")
    print(f"  Device={DEEP_DEVICE}  train_files={len(train_files)}  eval_files={len(eval_files)}")
    print(f"  max_epochs={max_epochs}  patience={patience}  amp={amp_enabled}")

    # ── Build model ──────────────────────────────────────────────────────
    model = build_deep_model(
        arch=arch,
        input_dim=len(DEEP_RAW_LOB_10_COLS),
        horizon_count=len(horizons),
        num_classes=3,
    ).to(DEEP_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg.get("lr", 3e-4)),
        weight_decay=float(cfg.get("weight_decay", 1e-4)),
    )
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    # ── Recover: resume from checkpoint if available ─────────────────────
    start_epoch   = 1
    best_score    = -1.0
    best_epoch    = 0
    best_metrics: dict = {}
    no_improve    = 0
    epoch_history: list = []

    best_eval_rows:   dict = {h: 0 for h in horizons}
    best_eval_counts: dict = {h: np.zeros(3, dtype=np.int64) for h in horizons}

    agg_train_rows:   dict = {h: 0 for h in horizons}
    agg_train_counts: dict = {h: np.zeros(3, dtype=np.int64) for h in horizons}
    total_bad_batches = 0
    total_train_days  = 0

    if not force_restart:
        ckpt = load_checkpoint(arch, suffix)
        if ckpt is not None:
            loaded = restore_weights_to_model(arch, model, label="current", suffix=suffix)
            if loaded:
                model.to(DEEP_DEVICE)
                start_epoch   = int(ckpt["epoch"]) + 1
                best_score    = float(ckpt["best_score"])
                best_epoch    = int(ckpt["best_epoch"])
                no_improve    = int(ckpt["no_improve"])
                epoch_history = list(ckpt.get("epoch_history", []))
                best_metrics  = dict(ckpt.get("best_metrics", {}))
                print(f"  ✓ Resumed from epoch {start_epoch - 1}  best_f1={best_score:.4f}  no_improve={no_improve}")
                if no_improve >= patience:
                    print(f"  Early stopping already triggered before restart. Returning saved best.")
                    # Already done — mark completed and return
                    save_checkpoint(arch, int(ckpt["epoch"]), model, optimizer,
                                    best_score, best_epoch, no_improve, epoch_history, best_metrics,
                                    completed=True, suffix=suffix)
                    return _finalize_arch(arch, cfg, model, best_metrics, best_score, best_epoch,
                                         epoch_history, train_files, eval_files, horizons,
                                         agg_train_rows, agg_train_counts, best_eval_rows, best_eval_counts,
                                         total_bad_batches, total_train_days, 0.0, max_epochs, patience, min_delta, suffix)
            else:
                print(f"  WARNING: Checkpoint found but weights missing for {arch}. Starting fresh.")

    if start_epoch > max_epochs:
        print(f"[{arch}] All {max_epochs} epochs already done. Marking complete.")
        save_checkpoint(arch, max_epochs, model, optimizer, best_score, best_epoch,
                        no_improve, epoch_history, best_metrics, completed=True, suffix=suffix)
        return _finalize_arch(arch, cfg, model, best_metrics, best_score, best_epoch,
                              epoch_history, train_files, eval_files, horizons,
                              agg_train_rows, agg_train_counts, best_eval_rows, best_eval_counts,
                              total_bad_batches, total_train_days, 0.0, max_epochs, patience, min_delta, suffix)

    # ── Training loop ────────────────────────────────────────────────────
    start_t = time.time()

    for epoch in range(start_epoch, max_epochs + 1):
        print(f"\n[{arch}] ── Epoch {epoch}/{max_epochs} ──────────────────")

        train_info = _train_epoch(
            model=model, optimizer=optimizer, scaler=scaler,
            cfg=cfg, train_files=train_files, horizons=horizons,
            epoch_idx=epoch, total_epochs=max_epochs, amp_enabled=amp_enabled, arch=arch,
        )

        eval_info = _eval_epoch(model=model, cfg=cfg, eval_files=eval_files, horizons=horizons, arch=arch)

        metrics = eval_info["metrics"]
        score   = _mean_macro_f1(metrics, horizons)
        improved = score > (best_score + min_delta)

        # Accumulate stats
        total_bad_batches += int(train_info["bad_batches"])
        total_train_days  += int(train_info["fitted_days"])
        for h in horizons:
            agg_train_rows[h]   += int(train_info["train_rows_seen"][h])
            agg_train_counts[h] += train_info["train_class_counts"][h]

        epoch_history.append({
            "epoch":          int(epoch),
            "mean_macro_f1":  float(score),
            "avg_train_loss": float(train_info["avg_train_loss"]),
            "bad_batches":    int(train_info["bad_batches"]),
            "improved":       bool(improved),
        })

        print(f"[{arch}] epoch={epoch}  mean_macro_f1={score:.6f}  best={best_score:.6f}  improved={improved}")

        if improved:
            best_score    = float(score)
            best_epoch    = int(epoch)
            best_metrics  = dict(metrics)
            best_eval_rows   = {h: int(eval_info["eval_rows_seen"][h]) for h in horizons}
            best_eval_counts = {h: eval_info["eval_class_counts"][h].copy() for h in horizons}
            save_best_weights(arch, model, suffix)
            no_improve = 0
        else:
            no_improve += 1

        # ── Save checkpoint after every epoch (key recovery point) ───────
        save_checkpoint(
            arch=arch, epoch=epoch, model=model, optimizer=optimizer,
            best_score=best_score, best_epoch=best_epoch, no_improve=no_improve,
            epoch_history=epoch_history, best_metrics=best_metrics,
            completed=False, suffix=suffix,
        )
        print(f"[{arch}] ✓ Checkpoint saved (epoch={epoch})")

        if no_improve >= patience:
            print(f"[{arch}] Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
            break

        _deep_cleanup_cuda()

    # ── Restore best weights ─────────────────────────────────────────────
    restored = restore_weights_to_model(arch, model, label="best", suffix=suffix)
    if not restored:
        print(f"[{arch}] WARNING: best weights not found, using current weights.")

    elapsed = time.time() - start_t

    # ── Mark completed ───────────────────────────────────────────────────
    save_checkpoint(
        arch=arch, epoch=best_epoch, model=model, optimizer=optimizer,
        best_score=best_score, best_epoch=best_epoch, no_improve=no_improve,
        epoch_history=epoch_history, best_metrics=best_metrics,
        completed=True, suffix=suffix,
    )

    return _finalize_arch(
        arch=arch, cfg=cfg, model=model, best_metrics=best_metrics,
        best_score=best_score, best_epoch=best_epoch, epoch_history=epoch_history,
        train_files=train_files, eval_files=eval_files, horizons=horizons,
        agg_train_rows=agg_train_rows, agg_train_counts=agg_train_counts,
        best_eval_rows=best_eval_rows, best_eval_counts=best_eval_counts,
        total_bad_batches=total_bad_batches, total_train_days=total_train_days,
        elapsed=elapsed, max_epochs=max_epochs, patience=patience, min_delta=min_delta, suffix=suffix,
    )


def _finalize_arch(
    arch, cfg, model, best_metrics, best_score, best_epoch, epoch_history,
    train_files, eval_files, horizons,
    agg_train_rows, agg_train_counts, best_eval_rows, best_eval_counts,
    total_bad_batches, total_train_days, elapsed, max_epochs, patience, min_delta, suffix,
) -> dict:
    """Save final weights + JSON result, return run_meta dict."""
    os.makedirs(cfg["weights_dir"], exist_ok=True)
    os.makedirs(cfg["results_dir"], exist_ok=True)

    weights_path = os.path.join(cfg["weights_dir"], f"{arch}{suffix}_weights.pt")
    torch.save({
        "architecture":   arch,
        "state_dict":     model.state_dict(),
        "input_dim":      len(DEEP_RAW_LOB_10_COLS),
        "horizons":       horizons,
        "seq_len":        int(cfg["seq_len"]),
        "alpha":          float(cfg["alpha"]),
        "device_trained": str(DEEP_DEVICE),
        "best_epoch":     int(best_epoch),
        "best_macro_f1":  float(best_score),
    }, weights_path)

    run_meta = {
        "timestamp":     pd.Timestamp.now().isoformat(),
        "mode":          "day_by_day_streaming_deep_stable_earlystop",
        "architecture":  arch,
        "horizons":      horizons,
        "seq_len":       int(cfg["seq_len"]),
        "alpha":         float(cfg["alpha"]),
        "train_config": {
            "max_epochs":       int(max_epochs),
            "patience":         int(patience),
            "min_delta":        float(min_delta),
            "batch_size":       int(cfg.get("batch_size", 256)),
            "lr":               float(cfg.get("lr", 3e-4)),
            "weight_decay":     float(cfg.get("weight_decay", 1e-4)),
            "grad_clip":        float(cfg.get("grad_clip", 1.0)),
            "amp":              bool(cfg.get("amp", False)),
            "label_smoothing":  float(cfg.get("label_smoothing", 0.0)),
            "device":           str(DEEP_DEVICE),
        },
        "early_stopping": {
            "best_epoch":       int(best_epoch),
            "best_mean_macro_f1": float(best_score),
            "epochs_ran":       int(len(epoch_history)),
        },
        "epoch_history":   epoch_history,
        "files": {
            "train_files": len(train_files),
            "eval_files":  len(eval_files),
            "days_fitted": int(total_train_days),
        },
        "train_rows_seen":    {f"h{h}": int(agg_train_rows[h]) for h in horizons},
        "eval_rows_seen":     {f"h{h}": int(best_eval_rows[h]) for h in horizons},
        "train_class_counts": {f"h{h}": [int(x) for x in agg_train_counts[h].tolist()] for h in horizons},
        "eval_class_counts":  {f"h{h}": [int(x) for x in best_eval_counts[h].tolist()] for h in horizons},
        "bad_batches_total": int(total_bad_batches),
        "test_metrics":      best_metrics,
        "model_path":        weights_path,
        "runtime_seconds":   round(elapsed, 2),
    }

    results_path = os.path.join(cfg["results_dir"], f"{arch}_results_day_streaming{suffix}.json")
    with open(results_path, "w") as f:
        json.dump(run_meta, f, indent=2)

    print(f"[{arch}] ✓ Weights  → {weights_path}")
    print(f"[{arch}] ✓ Results  → {results_path}")

    del model
    _deep_cleanup_cuda()
    return run_meta


print("train_one_architecture() ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 12 — RUN ALL ARCHITECTURES
#
# This cell is the main entry point. Run it once per session.
# On restart, it automatically:
#   - Skips completed architectures
#   - Resumes in-progress architectures from the last saved epoch
#
# To force a fresh restart for a specific arch, call:
#   train_one_architecture("arch_name", DEEP_CONFIG, force_restart=True, ...)
# ═══════════════════════════════════════════════════════════════════════════

deep_set_seed(int(DEEP_CONFIG["seed"]))

# Show status before starting
print_checkpoint_status()

all_runs: Dict[str, dict] = {}
pipeline_start = time.time()

for _arch in DEEP_CONFIG["run_architectures"]:
    print(f"\n{'▶'*3} Starting: {_arch}")
    try:
        _run_meta = train_one_architecture(
            arch=_arch,
            cfg=DEEP_CONFIG,
            max_epochs=DEEP_MAX_EPOCHS,
            patience=DEEP_EARLY_STOP_PATIENCE,
            min_delta=DEEP_EARLY_STOP_MIN_DELTA,
            force_restart=False,  # ← Set True to ignore checkpoint and retrain
        )
        all_runs[_arch] = _run_meta
    except Exception as _e:
        print(f"\n[ERROR] {_arch} failed with: {_e}")
        import traceback
        traceback.print_exc()
        print(f"Continuing to next architecture...")
        _deep_cleanup_cuda()
        continue

pipeline_elapsed = time.time() - pipeline_start

# ── Summary ──────────────────────────────────────────────────────────────
print("\n" + "═" * 80)
print("  PIPELINE COMPLETE")
print("═" * 80)
print(f"  Total time: {pipeline_elapsed/60:.1f} min")
print()

for _arch, _meta in all_runs.items():
    _f1   = _mean_macro_f1(_meta.get("test_metrics", {}), list(DEEP_CONFIG["horizons"]))
    _ep   = _meta.get("early_stopping", {}).get("best_epoch", _meta.get("best_epoch", "?"))
    _skip = _meta.get("skipped", False)
    _tag  = " (loaded from checkpoint)" if _skip else ""
    print(f"  {_arch:<30} mean_macro_f1={_f1:.4f}  best_epoch={_ep}{_tag}")

print()
print_checkpoint_status()

# Save summary
_summary = {
    "timestamp":     pd.Timestamp.now().isoformat(),
    "architectures": list(DEEP_CONFIG["run_architectures"]),
    "horizons":      list(DEEP_CONFIG["horizons"]),
    "max_epochs":    DEEP_MAX_EPOCHS,
    "patience":      DEEP_EARLY_STOP_PATIENCE,
    "runtime_seconds": round(pipeline_elapsed, 2),
    "results": {
        arch: {
            "best_mean_macro_f1": _mean_macro_f1(m.get("test_metrics", {}), list(DEEP_CONFIG["horizons"])),
            "best_epoch": m.get("early_stopping", {}).get("best_epoch", m.get("best_epoch", None)),
        }
        for arch, m in all_runs.items()
    },
}
_summary_path = os.path.join(RESULTS_DIR, "pipeline_summary.json")
with open(_summary_path, "w") as _f:
    json.dump(_summary, _f, indent=2)
print(f"Summary saved → {_summary_path}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 13 — Results & Per-Horizon Metrics
# Safe to run after Cell 12 or after a restart (loads from saved JSON).
# ═══════════════════════════════════════════════════════════════════════════

suffix  = str(DEEP_CONFIG.get("result_suffix", "_stable_es"))
archs   = DEEP_CONFIG["run_architectures"]
horizons = list(DEEP_CONFIG["horizons"])

print("\n" + "═" * 88)
print("  Results Summary")
print("═" * 88)

col_w = 10
header = f"{'Architecture':<30}" + "".join(f"{'h'+str(h)+' acc':>{col_w}}" for h in horizons) \
       + "".join(f"{'h'+str(h)+' f1m':>{col_w}}" for h in horizons) + f"{'mean f1m':>{col_w}}"
print(header)
print("-" * len(header))

for arch in archs:
    results_path = os.path.join(RESULTS_DIR, f"{arch}_results_day_streaming{suffix}.json")
    if not os.path.exists(results_path):
        # Try loading from all_runs if in memory
        meta = all_runs.get(arch) if "all_runs" in dir() else None
        if meta is None:
            print(f"  {arch:<30} — no results found")
            continue
    else:
        with open(results_path) as f:
            meta = json.load(f)

    metrics = meta.get("test_metrics", {})
    accs = [metrics.get(f"h{h}_accuracy", float("nan")) for h in horizons]
    f1s  = [metrics.get(f"h{h}_f1_macro",  float("nan")) for h in horizons]
    mean_f1 = float(np.nanmean(f1s)) if f1s else float("nan")

    row = f"{arch:<30}"
    row += "".join(f"{v:>{col_w}.4f}" for v in accs)
    row += "".join(f"{v:>{col_w}.4f}" for v in f1s)
    row += f"{mean_f1:>{col_w}.4f}"
    print(row)

print("═" * 88)

# Per-horizon detail
print("\nPer-horizon breakdown:")
for arch in archs:
    results_path = os.path.join(RESULTS_DIR, f"{arch}_results_day_streaming{suffix}.json")
    if not os.path.exists(results_path):
        continue
    with open(results_path) as f:
        meta = json.load(f)
    metrics = meta.get("test_metrics", {})
    best_ep = meta.get("early_stopping", {}).get("best_epoch", "?")
    print(f"\n  {arch}  (best_epoch={best_ep})")
    for h in horizons:
        acc = metrics.get(f"h{h}_accuracy", float("nan"))
        f1m = metrics.get(f"h{h}_f1_macro",  float("nan"))
        f1w = metrics.get(f"h{h}_f1_weighted", float("nan"))
        print(f"    h={h:>4}: acc={acc:.4f}  f1_macro={f1m:.4f}  f1_weighted={f1w:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 13b — R² Evaluation
#
# Computes R² metrics using the model's softmax class probabilities as
# "predicted values" against binary one-hot true labels.
#
# This measures how well the model's confidence correlates with outcomes —
# a complementary view to accuracy/F1 that is sensitive to calibration.
#
# R²_oos (out-of-sample) is the primary metric: it compares the model to
# a naive baseline that always predicts the class prior (mean probability).
# R²_oos > 0  → model beats the naive baseline.
# R²_oos ≈ 0  → model adds no value over the prior.
# R²_oos < 0  → model is worse than the naive baseline (badly miscalibrated).
# ═══════════════════════════════════════════════════════════════════════════

suffix   = str(DEEP_CONFIG.get("result_suffix", "_stable_es"))
horizons = list(DEEP_CONFIG["horizons"])
archs    = DEEP_CONFIG["run_architectures"]

def _load_model_for_eval(arch: str, cfg: dict, suffix: str) -> "nn.Module | None":
    weights_path = os.path.join(cfg["weights_dir"], f"{arch}{suffix}_weights.pt")
    if not os.path.exists(weights_path):
        # Try best checkpoint weights
        best_path = os.path.join(cfg["weights_dir"], f"{arch}{suffix}_best_weights.pt")
        if not os.path.exists(best_path):
            print(f"  [{arch}] No weights found at {weights_path}")
            return None
        weights_path = best_path
    try:
        ckpt = torch.load(weights_path, map_location="cpu")
        model = build_deep_model(arch=arch, input_dim=len(DEEP_RAW_LOB_10_COLS),
                                  horizon_count=len(horizons), num_classes=3)
        sd = ckpt.get("state_dict", ckpt)
        model.load_state_dict(sd)
        model.eval()
        return model
    except Exception as e:
        print(f"  [{arch}] Failed to load weights: {e}")
        return None


def _collect_probs_and_labels(
    model: "nn.Module",
    cfg: dict,
    eval_files: List[Tuple[str, str]],
    horizons: List[int],
    max_batches: int = 50,
) -> Tuple["np.ndarray | None", "np.ndarray | None"]:
    """Run model on eval files, collect softmax probs and true labels."""
    all_probs:  List[np.ndarray] = []
    all_labels: List[np.ndarray] = []
    n_batches = 0

    model.to(DEEP_DEVICE)
    with torch.no_grad():
        for ticker, path in eval_files:
            ds, _ = _deep_build_day_dataset(path, cfg, is_train=False)
            if ds is None:
                continue
            loader = _deep_make_loader(ds, cfg, is_train=False)
            for xb, yb in loader:
                xb = torch.nan_to_num(
                    xb.to(DEEP_DEVICE, non_blocking=True),
                    nan=0.0, posinf=0.0, neginf=0.0
                ).clamp(-10.0, 10.0)
                logits = model(xb)                           # (B, H, 3)
                probs  = torch.softmax(logits, dim=-1)       # (B, H, 3)
                all_probs.append(probs.detach().cpu().numpy())
                all_labels.append(yb.numpy())
                n_batches += 1
                del xb, logits, probs
                if n_batches >= max_batches:
                    break
            del loader, ds
            _deep_cleanup_cuda()
            if n_batches >= max_batches:
                break

    if not all_probs:
        return None, None
    return np.concatenate(all_probs, axis=0), np.concatenate(all_labels, axis=0)


print("═" * 65)
print("  R² Evaluation (per horizon, per architecture)")
print("═" * 65)

# Collect eval files (same split as training)
_eval_tickers = _deep_resolve_tickers(DEEP_CONFIG)
_eval_files_by_ticker = _deep_collect_files_by_ticker(
    DEEP_CONFIG["data_dir"], _eval_tickers,
    int(DEEP_CONFIG.get("max_files_per_ticker", 0))
)
_, _eval_files = _deep_split_train_eval_files(
    _eval_files_by_ticker, float(DEEP_CONFIG.get("train_file_fraction", 0.8))
)

for _arch in archs:
    print(f"\n  [{_arch}]")
    _model = _load_model_for_eval(_arch, DEEP_CONFIG, suffix)
    if _model is None:
        print("    No weights available — train first (Cell 12)")
        continue

    _probs, _y_true = _collect_probs_and_labels(_model, DEEP_CONFIG, _eval_files, horizons)
    del _model
    _deep_cleanup_cuda()

    if _probs is None:
        print("    No eval data available")
        continue

    # Compute R² for each horizon using:
    #   y_pred  = probability of the true class  (model confidence)
    #   y_true  = 1.0 (the true class always "happened")
    #   Also compute R² on predicted-class probability vs one-hot label
    _n_feats = len(DEEP_RAW_LOB_10_COLS)
    _r2_results = {}
    for _i, _h in enumerate(horizons):
        _y_h   = _y_true[:, _i].astype(np.int64)       # (N,)  values in {0,1,2}
        _p_h   = _probs[:, _i, :]                        # (N, 3) softmax probs

        # One-hot encode true labels
        _y_oh  = np.eye(3)[_y_h].astype(np.float64)     # (N, 3)

        # R² of each class probability vs its one-hot column
        _r2_per_cls = []
        for _c in range(3):
            _r2_per_cls.append(compute_r2(_y_oh[:, _c], _p_h[:, _c].astype(np.float64)))

        _r2_mean   = float(np.mean(_r2_per_cls))
        _r2_oos    = compute_oos_r2(_y_oh.ravel(), _p_h.ravel().astype(np.float64))
        _r2_adj    = compute_adjusted_r2(_y_oh.ravel(), _p_h.ravel().astype(np.float64), _n_feats)

        _r2_results[f"h{_h}"] = {"r2_mean": _r2_mean, "r2_oos": _r2_oos, "r2_adj": _r2_adj,
                                  "r2_per_class": _r2_per_cls}

        _cls_names = ["DOWN", "STAT", "UP"]
        print(f"    h={_h:>4}  R²_mean={_r2_mean:+.4f}  R²_oos={_r2_oos:+.4f}  "
              f"R²_adj={_r2_adj:+.4f}", end="")
        print(f"  | per-class: {' '.join(f'{_cls_names[c]}={_r2_per_cls[c]:+.3f}' for c in range(3))}")

    print_r2_table(_r2_results, horizons)

    # Save R² results to disk
    _r2_path = os.path.join(RESULTS_DIR, f"{_arch}{suffix}_r2_metrics.json")
    with open(_r2_path, "w") as _f:
        json.dump({
            "architecture": _arch,
            "timestamp": pd.Timestamp.now().isoformat(),
            "horizons": horizons,
            "r2_results": {
                k: {**v, "r2_per_class": [float(x) for x in v["r2_per_class"]]}
                for k, v in _r2_results.items()
            },
            "n_samples": int(_probs.shape[0]),
        }, _f, indent=2)
    print(f"    ✓ R² metrics saved → {_r2_path}")
    del _probs, _y_true, _r2_results
    gc.collect()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 13c — Class Imbalance Report  (post-training analysis)
#
# Shows class distribution seen during training + SMOTE install helper.
# ═══════════════════════════════════════════════════════════════════════════
import subprocess

def _try_install_imblearn():
    try:
        import imblearn
        return True
    except ImportError:
        pass
    print("imbalanced-learn not found. Installing ...")
    try:
        subprocess.run(["pip", "install", "imbalanced-learn", "-q"], check=True)
        print("✓ imbalanced-learn installed. Re-run this cell to confirm.")
        return True
    except Exception as e:
        print(f"Install failed: {e}. Run:  !pip install imbalanced-learn")
        return False

_imblearn_ok = _try_install_imblearn()

# ── Print imbalance from training run results ─────────────────────────────
print("\n" + "═"*60)
print("  Imbalance Report (from training results)")
print("═"*60)
suffix = str(DEEP_CONFIG.get("result_suffix", "_stable_es"))
for _arch in DEEP_CONFIG["run_architectures"]:
    _rp = os.path.join(RESULTS_DIR, f"{_arch}_results_day_streaming{suffix}.json")
    if not os.path.exists(_rp):
        print(f"  {_arch}: no results yet")
        continue
    with open(_rp) as _f:
        _meta = json.load(_f)
    print(f"\n  {_arch}")
    for _hk, _counts in _meta.get("train_class_counts", {}).items():
        _total = sum(_counts)
        if _total == 0: continue
        _pcts = [f"{100.0*c/_total:.1f}%" for c in _counts]
        _ir   = max(_counts) / max(min(c for c in _counts if c>0), 1)
        print(f"    {_hk}: DOWN={_pcts[0]}  STAT={_pcts[1]}  UP={_pcts[2]}  "
              f"imbalance_ratio={_ir:.1f}x")

print()
if not DEEP_CONFIG.get("enable_smote") and _imblearn_ok:
    print("💡 Tip: set enable_smote=True in Cell 4 to apply SMOTE oversampling.")
    print("   Recommended when imbalance_ratio > 5×.")
elif DEEP_CONFIG.get("enable_smote") and _imblearn_ok:
    print(f"✓ SMOTE is enabled (method={DEEP_CONFIG.get('smote_method','smote')}).")
elif not _imblearn_ok:
    print("⚠ Install imbalanced-learn to use SMOTE: !pip install imbalanced-learn")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 14 — Epoch History & Training Curves
# ═══════════════════════════════════════════════════════════════════════════

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not available — printing text curves only")

suffix = str(DEEP_CONFIG.get("result_suffix", "_stable_es"))
archs  = DEEP_CONFIG["run_architectures"]

for arch in archs:
    results_path = os.path.join(RESULTS_DIR, f"{arch}_results_day_streaming{suffix}.json")
    if not os.path.exists(results_path):
        print(f"  {arch}: no results file yet")
        continue
    with open(results_path) as f:
        meta = json.load(f)
    history = meta.get("epoch_history", [])
    if not history:
        print(f"  {arch}: no epoch history")
        continue

    epochs = [h["epoch"] for h in history]
    f1s    = [h["mean_macro_f1"] for h in history]
    losses = [h["avg_train_loss"] for h in history]

    print(f"\n  {arch} — epoch history:")
    for ep, f1, loss in zip(epochs, f1s, losses):
        marker = " ◀ best" if f1 == max(f1s) else ""
        print(f"    ep={ep:>3}  mean_macro_f1={f1:.4f}  avg_train_loss={loss:.4f}{marker}")

    if HAS_MPL:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
        ax1.plot(epochs, f1s, marker="o")
        ax1.set_title(f"{arch} — mean macro F1")
        ax1.set_xlabel("epoch"); ax1.set_ylabel("f1")
        ax2.plot(epochs, losses, marker="o", color="orange")
        ax2.set_title(f"{arch} — avg train loss")
        ax2.set_xlabel("epoch"); ax2.set_ylabel("loss")
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, f"{arch}_training_curve.png"), dpi=100)
        plt.show()
        print(f"  Saved training curve → {arch}_training_curve.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 15 — (OPTIONAL) Reset / Force-Restart a Specific Architecture
#
# Run this cell only if you want to ERASE the checkpoint for a specific
# architecture and start it fresh next time Cell 12 runs.
#
# Usage:
#   RESET_ARCH = "dilated_transformer"  # change to your target
#   Then uncomment and run the block below.
# ═══════════════════════════════════════════════════════════════════════════

# RESET_ARCH = "dilated_transformer"   # ← change this
# _suffix = str(DEEP_CONFIG.get("result_suffix", "_stable_es"))
#
# _paths_to_remove = [
#     _ckpt_path(RESET_ARCH, _suffix),
#     _ckpt_weights_path(RESET_ARCH, "current", _suffix),
#     _ckpt_weights_path(RESET_ARCH, "best",    _suffix),
# ]
# for _p in _paths_to_remove:
#     if os.path.exists(_p):
#         os.remove(_p)
#         print(f"Deleted: {_p}")
#     else:
#         print(f"Not found (skipped): {_p}")
# print(f"Reset complete for {RESET_ARCH}. Re-run Cell 12 to retrain.")

print("Cell 15 — reset utility (commented out by default). Uncomment to use.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 16 — Export All Results to a Single ZIP
#
# Bundles every result file into one ZIP for easy single-click download.
# Contents:
#   results/          → all JSON metrics, PNG plots, summaries
#   model_weights/    → final *_weights.pt files only (not recovery checkpoints)
#
# In Colab  → triggers an automatic browser download.
# Locally   → prints the path to the ZIP file.
#
# Run any time after training (or between runs to save intermediate results).
# ═══════════════════════════════════════════════════════════════════════════

import zipfile
import shutil

# ── Config ────────────────────────────────────────────────────────────────
_timestamp   = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
_zip_name    = f"multi_horizon_ofi_results_{_timestamp}.zip"
_zip_path    = str(PROJECT_ROOT / _zip_name)

# Which model_weights files to include (final weights only, skip recovery files)
_WEIGHTS_INCLUDE_PATTERNS = [
    "*_stable_es_weights.pt",    # final best weights from new pipeline
    "*.pt",                      # any other final weights (legacy)
]
_WEIGHTS_EXCLUDE_PATTERNS = [
    "*_current_weights.pt",      # mid-training recovery
    "*_best_weights.pt",         # mid-training best copy (redundant with final)
]

def _is_final_weight(filename: str) -> bool:
    name = os.path.basename(filename)
    if any(name.endswith(p.replace("*", "")) for p in _WEIGHTS_EXCLUDE_PATTERNS
           if p.startswith("*")):
        return False
    return True

# ── Build file list ────────────────────────────────────────────────────────
_to_zip: List[Tuple[str, str]] = []   # (abs_path, archive_name)

# 1. Everything in results/
_results_dir = str(PROJECT_ROOT / "results")
if os.path.isdir(_results_dir):
    for root, dirs, files in os.walk(_results_dir):
        for fname in files:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, str(PROJECT_ROOT))
            _to_zip.append((fpath, arcname))

# 2. Final model weights only (skip checkpoints)
_weights_dir = str(PROJECT_ROOT / "model_weights")
if os.path.isdir(_weights_dir):
    for fname in os.listdir(_weights_dir):
        fpath = os.path.join(_weights_dir, fname)
        if os.path.isfile(fpath) and _is_final_weight(fpath):
            arcname = os.path.join("model_weights", fname)
            _to_zip.append((fpath, arcname))

# 3. The pipeline summary JSON (if it exists at root level)
for _extra in ["pipeline_summary.json"]:
    _ep = str(PROJECT_ROOT / "results" / _extra)
    if os.path.exists(_ep) and (_ep, f"results/{_extra}") not in _to_zip:
        _to_zip.append((_ep, f"results/{_extra}"))

# ── Write ZIP ─────────────────────────────────────────────────────────────
print(f"Creating {_zip_name} ...")
_skipped = 0
_added   = 0
_total_bytes = 0

with zipfile.ZipFile(_zip_path, "w", compression=zipfile.ZIP_DEFLATED,
                     compresslevel=6) as _zf:
    for _src, _arcname in _to_zip:
        if not os.path.exists(_src):
            _skipped += 1
            continue
        _zf.write(_src, _arcname)
        _sz = os.path.getsize(_src)
        _total_bytes += _sz
        _added += 1
        print(f"  + {_arcname:<70} ({_sz/1024:.0f} KB)")

_zip_size = os.path.getsize(_zip_path)
print(f"\n{'═'*60}")
print(f"  ZIP created : {_zip_path}")
print(f"  Files added : {_added}  (skipped={_skipped})")
print(f"  Raw size    : {_total_bytes/1024/1024:.1f} MB")
print(f"  ZIP size    : {_zip_size/1024/1024:.1f} MB")
print(f"{'═'*60}\n")

# ── Colab download ────────────────────────────────────────────────────────
if IN_COLAB:
    from google.colab import files  # type: ignore
    print("Starting download ...")
    files.download(_zip_path)
    print(f"✓ Download triggered: {_zip_name}")
else:
    print(f"Local mode — ZIP saved at:")
    print(f"  {_zip_path}")
    print(f"\nTo copy to another location:")
    print(f"  cp \"{_zip_path}\" /your/destination/")
